# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/tmp/ipykernel_4956/3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [ ]:
pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [ ]:
pip install langchain_chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [ ]:
pip install langchain_huggingface

In [ ]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
# We are using a local, low-cost model through Ollama

MODEL = "ollama/llama3.2"
db_name = "vector_db"

from openai import OpenAI

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print(f"Using Ollama model: {MODEL}")

Using Ollama model: ollama/llama3.2


In [ ]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


Found 0 files in the knowledge base
Total characters in knowledge base: 0


In [ ]:
import zipfile

# Extract evaluation
with zipfile.ZipFile("/content/evaluation.zip", "r") as zip_ref:
    zip_ref.extractall("/")

# Extract implementation
with zipfile.ZipFile("/content/implementation.zip", "r") as zip_ref:
    zip_ref.extractall("/")

# Extract knowledge base
with zipfile.ZipFile("/content/knowledge-base.zip", "r") as zip_ref:
    zip_ref.extractall("/")

print("All three ZIP files extracted successfully!")

All three ZIP files extracted successfully!


In [ ]:
# We are using a local, low-cost model through Ollama

MODEL = "ollama/llama3.2"
db_name = "vector_db"

from openai import OpenAI

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print(f"Using Ollama model: {MODEL}")

Using Ollama model: ollama/llama3.2


In [ ]:
# How many tokens in all the documents?

import tiktoken

# tiktoken doesn't have a direct Llama 3.2 mapping.
# Use an explicit tokenizer for an approximate token count.
encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(entire_knowledge_base)

token_count = len(tokens)

print(f"Total tokens: {token_count:,}")

Total tokens: 0


In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("/knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.

After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.

### Our Culture' metadata={'source': '/knowledge-base/company/careers.md', 'doc_type': 'company'}


In [ ]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 413 documents


In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


In [ ]:
pip install litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL = "llama3.2"


# ============================================================
# VECTOR DATABASE CONFIGURATION
# ============================================================

DB_NAME = "preprocessed_db"

collection_name = "docs"


# ============================================================
# EMBEDDING MODEL
# ============================================================

# We are using the same embedding model that created
# your current Chroma database.
embedding_model = "all-MiniLM-L6-v2"


# ============================================================
# KNOWLEDGE BASE
# ============================================================

KNOWLEDGE_BASE_PATH = Path("/knowledge-base")

AVERAGE_CHUNK_SIZE = 500

In [ ]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [ ]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [ ]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: /knowledge-base/company/careers.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 12 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Careers at Insurellm

## Why Join Insurellm?



In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: /knowledge-base/company/careers.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 12 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\

In [24]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [27]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [28]:
import time
time.sleep(5)

In [29]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-08-08T18:44:36.164932957Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [26]:
!ollama run llama3.2 "Reply with only OK"

Error: could not connect to ollama server, run 'ollama serve' to start it


In [30]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [31]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [32]:
import time
time.sleep(5)

In [33]:
!ollama --version

ollama version is 0.32.6


In [34]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [35]:
!which ollama

/usr/local/bin/ollama


In [36]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [37]:
!which ollama
!ollama --version

/usr/local/bin/ollama
ollama version is 0.32.6


In [38]:
!ls -l /usr/local/bin/ollama
!ls -l /usr/bin/ollama

-rwxr-xr-x 1 root root 38039440 Aug  5 17:29 /usr/local/bin/ollama
ls: cannot access '/usr/bin/ollama': No such file or directory


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 137 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (701 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!which ollama

/usr/local/bin/ollama


In [ ]:
!ollama --version

In [124]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [125]:
import time
time.sleep(5)

In [103]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-08-08T17:48:02.524198559Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [126]:
!ollama pull llama3.2

In [127]:
!ollama list

NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    3 seconds ago    


In [ ]:
!ollama run llama3.2 "Reply with only OK"

OK



In [128]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-08-08T18:44:36.164932957Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [ ]:
!ps aux | grep ollama

root       35463  0.2  0.2 1960040 38576 ?       Sl   17:47   0:02 ollama serve
root       39354  0.0  0.0   7372  3596 ?        S    18:03   0:00 /bin/bash -c ps aux | grep ollama
root       39356  0.0  0.0   6480  2424 ?        S    18:03   0:00 grep ollama


In [ ]:
!tail -50 /tmp/ollama.log

slot print_timing: id  0 | task 3 | n_decoded =    362, tg =   1.53 t/s, tg_3s =   1.33 t/s
slot print_timing: id  0 | task 3 | n_decoded =    366, tg =   1.53 t/s, tg_3s =   1.16 t/s
slot print_timing: id  0 | task 3 | n_decoded =    372, tg =   1.53 t/s, tg_3s =   1.81 t/s
slot print_timing: id  0 | task 3 | n_decoded =    377, tg =   1.53 t/s, tg_3s =   1.49 t/s
slot print_timing: id  0 | task 3 | n_decoded =    382, tg =   1.53 t/s, tg_3s =   1.38 t/s
slot print_timing: id  0 | task 3 | n_decoded =    386, tg =   1.52 t/s, tg_3s =   1.13 t/s
slot print_timing: id  0 | task 3 | n_decoded =    391, tg =   1.52 t/s, tg_3s =   1.50 t/s
slot print_timing: id  0 | task 3 | n_decoded =    397, tg =   1.52 t/s, tg_3s =   1.74 t/s
slot print_timing: id  0 | task 3 | n_decoded =    402, tg =   1.53 t/s, tg_3s =   1.59 t/s
slot print_timing: id  0 | task 3 | n_decoded =    406, tg =   1.52 t/s, tg_3s =   1.24 t/s
slot print_timing: id  0 | task 3 | n_decoded =    411, tg =   1.52 t/s, tg_3s =

In [122]:
!curl -s http://127.0.0.1:11434/api/tags

In [39]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Reply only with OK",
        "stream": False,
        "options": {
            "num_predict": 10
        }
    },
    timeout=120
)

print(response.json()["response"])

OK


In [42]:
import sys

# Make the root directory available for imports
if "/" not in sys.path:
    sys.path.insert(0, "/")

# Check that the folder exists
!ls -la /evaluation

total 72
drwxr-xr-x 3 root root  4096 Aug  8 18:35 .
drwxr-xr-x 1 root root  4096 Aug  8 15:56 ..
-rw-r--r-- 1 root root 12345 Aug  8 18:25 eval.py
-rw-r--r-- 1 root root     0 Aug  8 18:35 __init__.py
drwxr-xr-x 2 root root  4096 Aug  8 18:35 __pycache__
-rw-r--r-- 1 root root   906 Aug  8 15:56 test.py
-rw-r--r-- 1 root root 38644 Aug  8 15:56 tests.jsonl


In [43]:
import evaluation

print("evaluation package:", evaluation.__file__)

evaluation package: /evaluation/__init__.py


In [44]:
from evaluation.eval import evaluate_answer, evaluate_retrieval
import evaluation.eval

print("MODEL:", evaluation.eval.MODEL)
print("OLLAMA:", evaluation.eval.OLLAMA_API_BASE)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MODEL: ollama/llama3.2
OLLAMA: http://127.0.0.1:11434


In [45]:
import sys

for module in list(sys.modules):
    if module.startswith("evaluation"):
        del sys.modules[module]

from evaluation.eval import evaluate_answer, evaluate_retrieval
import evaluation.eval

print("MODEL:", evaluation.eval.MODEL)
print("OLLAMA:", evaluation.eval.OLLAMA_API_BASE)

MODEL: ollama/llama3.2
OLLAMA: http://127.0.0.1:11434


In [46]:
import sys

sys.path.insert(0, "/")

print(sys.path[:3])

['/', '/', '/content']


In [47]:
!ls -la /evaluation

total 72
drwxr-xr-x 3 root root  4096 Aug  8 18:35 .
drwxr-xr-x 1 root root  4096 Aug  8 15:56 ..
-rw-r--r-- 1 root root 12345 Aug  8 18:25 eval.py
-rw-r--r-- 1 root root     0 Aug  8 18:35 __init__.py
drwxr-xr-x 2 root root  4096 Aug  8 18:35 __pycache__
-rw-r--r-- 1 root root   906 Aug  8 15:56 test.py
-rw-r--r-- 1 root root 38644 Aug  8 15:56 tests.jsonl


In [48]:
import sys

if "/" not in sys.path:
    sys.path.insert(0, "/")

print(sys.path[:3])

['/', '/', '/content']


In [49]:
!touch /evaluation/__init__.py

In [50]:
from evaluation.eval import evaluate_answer, evaluate_retrieval

print("✅ evaluation imported successfully")

✅ evaluation imported successfully


In [117]:
!pip install -q langchain-ollama

In [51]:
import sys

for module in list(sys.modules):
    if module.startswith("implementation") or module.startswith("evaluation"):
        del sys.modules[module]

In [52]:
from implementation.answer import answer_question, fetch_context

print("✅ answer.py loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ answer.py loaded


In [53]:
from evaluation.eval import evaluate_answer, evaluate_retrieval

print("✅ eval.py loaded")

✅ eval.py loaded


In [14]:
!grep -R -n "def process_document" /content /evaluation /implementation 2>/dev/null | head -20

In [17]:
!pip install -q langchain-text-splitters

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

def process_document(doc):
    """
    Split one Document into smaller chunks.
    """

    return text_splitter.split_documents([doc])

In [19]:
process_document(documents[0])


[Document(metadata={'source': '/knowledge-base/company/careers.md'}, page_content="# Careers at Insurellm\n\n## Why Join Insurellm?\n\nAt Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.\n\nAfter reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.\n\n### Our Culture"),
 Document(metadata={'source': '/knowledge-base/company/careers.md'}, page_content='### Our Culture\n\nWe live by our core values every day:\n- *

In [55]:
import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Who is Avery Lancaster? Answer in one short sentence.",
        "stream": False,
        "options": {
            "num_predict": 50
        }
    },
    timeout=120
)

print(response.json()["response"])

I couldn't find any notable or well-known individual by the name of Avery Lancaster, could you provide more context or information about who Avery Lancaster is?


In [56]:
docs = fetch_context("Who is Averi Lancaster?")

print("Documents retrieved:", len(docs))

for i, doc in enumerate(docs):
    print(f"\n--- DOCUMENT {i+1} ---")
    print(doc.page_content[:1000])

Documents retrieved: 10

--- DOCUMENT 1 ---
Emily Carter exemplifies the kind of talent that drives Insurellm's success and is an invaluable asset to the company.

--- DOCUMENT 2 ---
## Other HR Notes
- **Professional Development:** Completed Product Leadership Certification from Pragmatic Institute (2021). Regular attendee at ProductCon and InsurTech conferences.
- **Mentorship:** Currently mentors two Associate Product Managers and actively participates in the Women in Tech initiative at Insurellm.
- **Skills:** Expert in Agile/Scrum methodologies, JIRA, Figma, SQL, and product analytics tools.
- **Feedback:** Known for her strategic thinking and ability to balance customer needs with business objectives. Strong communicator who bridges technical and business stakeholders effectively.

--- DOCUMENT 3 ---
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Sa

In [133]:
answer, docs = answer_question(
    "Who is Avery Lancaster?",
    []
)

print(answer)

Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, a leading Insurance Tech provider. She has been instrumental in guiding the company to its current position in the industry, known for her innovative leadership strategies and risk management expertise.


In [7]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [6]:
from pathlib import Path
from langchain_core.documents import Document

knowledge_base_path = Path("/knowledge-base")

documents = []

for file_path in knowledge_base_path.rglob("*"):
    if file_path.is_file():
        try:
            text = file_path.read_text(encoding="utf-8")
            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": str(file_path)}
                )
            )
        except UnicodeDecodeError:
            print(f"Skipped non-text file: {file_path}")

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [59]:
from tqdm import tqdm

In [61]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

def process_document(document):
    return text_splitter.split_documents([document])

In [62]:
from tqdm import tqdm

def create_chunks(documents):
    chunks = []

    for doc in tqdm(documents):
        chunks.extend(process_document(doc))

    return chunks

In [63]:
chunks = create_chunks(documents)

print(f"Created {len(chunks)} chunks")

100%|██████████| 76/76 [00:00<00:00, 6811.26it/s]

Created 413 chunks


In [64]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [00:00<00:00, 8193.68it/s]


In [ ]:
print(len(chunks))

zx

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [66]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

DB_NAME = "/content/vector_db"

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

def create_embeddings(chunks):
    """
    Create a Chroma vector database from the document chunks.
    """

    # Remove old database if it exists
    if os.path.exists(DB_NAME):
        import shutil
        shutil.rmtree(DB_NAME)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

    print(
        f"Vector store created with "
        f"{vectorstore._collection.count()} documents"
    )

    return vectorstore

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [68]:
import os
import shutil

DB_NAME = "/content/vector_db_new"

if os.path.exists(DB_NAME):
    shutil.rmtree(DB_NAME)

os.makedirs(DB_NAME, exist_ok=True)

print("Database folder ready:", DB_NAME)

Database folder ready: /content/vector_db_new


In [69]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

def create_embeddings(chunks):

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

    print(
        "Vector store created with",
        vectorstore._collection.count(),
        "documents"
    )

    return vectorstore

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [70]:
vectorstore = create_embeddings(chunks)

Vector store created with 413 documents


In [71]:
create_embeddings(chunks)

Vector store created with 826 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [73]:
import numpy as np
from chromadb import PersistentClient

In [74]:
DB_NAME = "/content/vector_db_new"
collection_name = "docs"

In [75]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [79]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [80]:
print("Number of vectors:", vectors.shape[0])

Number of vectors: 0


In [83]:
print("vectors shape:", vectors.shape)
print("number of samples:", len(vectors))

vectors shape: (0,)
number of samples: 0


In [86]:
print("Number of vectors:", vectors.shape[0])

Number of vectors: 0


In [89]:
n_samples = vectors.shape[0]

if n_samples <= 2:
    print(f"Only {n_samples} vectors available. Cannot run t-SNE.")
else:
    perplexity = min(30, n_samples - 1)

    print("Number of vectors:", n_samples)
    print("Perplexity:", perplexity)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=42
    )

    reduced_vectors = tsne.fit_transform(vectors)

    print("✅ t-SNE completed")

Only 0 vectors available. Cannot run t-SNE.


In [90]:
n_samples = vectors.shape[0]

if n_samples <= 2:
    print(f"Only {n_samples} vectors available. Cannot run t-SNE.")
else:
    perplexity = min(30, n_samples - 1)

    print("Number of vectors:", n_samples)
    print("Perplexity:", perplexity)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=42
    )

    reduced_vectors = tsne.fit_transform(vectors)

    print("✅ t-SNE completed")

Only 0 vectors available. Cannot run t-SNE.


## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [92]:
from pydantic import BaseModel, Field

In [93]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [94]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [97]:
def fetch_context_unranked(question):
    query = embeddings.embed_query(question)

    results = collection.query(
        query_embeddings=[query],
        n_results=RETRIEVAL_K
    )

    chunks = []

    for i, document in enumerate(results["documents"][0]):
        chunks.append(
            Document(
                page_content=document,
                metadata=results["metadatas"][0][i]
                if results.get("metadatas")
                else {}
            )
        )

    return chunks

In [98]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [99]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [101]:
from litellm import completion

MODEL = "ollama/llama3.2"
OLLAMA_API_BASE = "http://127.0.0.1:11434"

In [103]:
def rerank(question, chunks):
    messages = [
        {
            "role": "system",
            "content": """
You are a document relevance ranking system.

Rank the provided chunks from most relevant to least relevant
for the question.

Return ONLY a JSON object in this format:

{
    "order": [1, 2, 3]
}

IMPORTANT:
- Use only chunk numbers that actually exist.
- If there are 3 chunks, only use 1, 2, and 3.
- Do not invent chunk numbers.
"""
        },
        {
            "role": "user",
            "content": f"""
Question:
{question}

Chunks:

""" + "\n\n".join(
                f"Chunk {i + 1}:\n{chunk.page_content}"
                for i, chunk in enumerate(chunks)
            )
        }
    ]

    response = completion(
        model=MODEL,
        messages=messages,
        api_base=OLLAMA_API_BASE,
        response_format=RankOrder,
        temperature=0,
        max_tokens=100,
        timeout=300
    )

    reply = response.choices[0].message.content

    order = RankOrder.model_validate_json(reply).order

    print("Llama returned order:", order)

    # Keep only valid chunk numbers
    valid_order = [
        i for i in order
        if 1 <= i <= len(chunks)
    ]

    # Add any chunks Llama forgot to include
    missing = [
        i for i in range(1, len(chunks) + 1)
        if i not in valid_order
    ]

    valid_order.extend(missing)

    print("Final order:", valid_order)

    return [
        chunks[i - 1]
        for i in valid_order
    ]

In [107]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [105]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [106]:
reranked = rerank(question, chunks)

Llama returned order: [1]
Final order: []


In [108]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [109]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [110]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [111]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [112]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [113]:
rewrite_query("Who won the IIOTY award?", [])

'Who won IIOTY award?'

In [114]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [115]:
answer_question("Who won the IIOTY award?", [])

Who won IIOTY award?
Llama returned order: [2]
Final order: []


("I don't have information on who won the IIOTY (International IoT Young Technology Award). Insurellm focuses more on insurance-related topics rather than tech awards. If you're looking for information on a specific year's winner, I recommend checking the official website of the International IoT Society or other relevant sources.",
 [])

In [116]:
answer_question("Who went to Manchester University?", [])

Manchester University staff?
Llama returned order: [1, 2]
Final order: []


("I'm not aware of any information about individuals attending Manchester University from Insurellm's knowledge base. However, I can tell you that Manchester University is a well-known institution in the UK, founded in 1824 as the Chemical Society's Laboratory and later becoming a full-fledged university. If you're looking for specific information about alumni or notable figures who have attended Manchester University, I'd be happy to try and help further.",
 [])